# Unit Address Prediction (LightGBM + SHAP)

Input a detailed unit address (e.g. block + street + unit).
The notebook auto-completes required features from API/data sources, predicts price with saved `lightgbm_tuned_v2`, and explains top 20 SHAP factors.

In [2]:
import hashlib
import json
import re
from datetime import datetime
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import requests
from sklearn.neighbors import BallTree

ROOT = Path('/Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens')
RAW_DIR = ROOT / '01_data_layer' / 'raw'
GEO_DIR = RAW_DIR / 'google_geo'
SCHOOL_DIR = RAW_DIR / 'schools'
FEATURE_OUTPUT = ROOT / '02_feature_layer' / 'training' / 'outputs'
TRAIN_OUTPUT = ROOT / '03_ml_layer' / 'training' / 'outputs'
TEST_OUTPUT = ROOT / '03_ml_layer' / 'test' / 'outputs'
TEST_OUTPUT.mkdir(parents=True, exist_ok=True)

RUN_DATE = datetime.now().strftime('%Y%m%d')
SEED = 42
session = requests.Session()

ONEMAP_SEARCH_ENDPOINT = 'https://www.onemap.gov.sg/api/common/elastic/search'

print('TRAIN_OUTPUT:', TRAIN_OUTPUT)
print('FEATURE_OUTPUT:', FEATURE_OUTPUT)
print('TEST_OUTPUT:', TEST_OUTPUT)

TRAIN_OUTPUT: /Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens/03_ml_layer/training/outputs
FEATURE_OUTPUT: /Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens/02_feature_layer/training/outputs
TEST_OUTPUT: /Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens/03_ml_layer/test/outputs


In [3]:
def latest_file(folder: Path, pattern: str) -> Path:
    files = sorted(folder.glob(pattern))
    if not files:
        raise FileNotFoundError(f'No file for pattern {pattern} in {folder}')
    return files[-1]

def normalize_address(v: str) -> str:
    s = str(v).upper().strip()
    s = re.sub(r',\s*SINGAPORE\s*\d*$', '', s)
    s = re.sub(r'\s+#\d{1,2}-\d{1,4}[A-Z]?\s*$', '', s)
    s = re.sub(r'\s+', ' ', s).strip()
    parts = s.split(' ')
    if parts and parts[0].isdigit():
        parts[0] = str(int(parts[0]))
    return ' '.join(parts)

def compact(v: str) -> str:
    return re.sub(r'[^A-Z0-9]', '', str(v).upper())

def parse_unit_floor(v: str):
    m = re.search(r'#(\d{1,2})-\d{1,4}[A-Z]?$', str(v).upper().strip())
    if not m:
        return None
    return int(m.group(1))

def onemap_geocode(query: str):
    try:
        r = session.get(ONEMAP_SEARCH_ENDPOINT, params={
            'searchVal': query,
            'returnGeom': 'Y',
            'getAddrDetails': 'Y',
            'pageNum': 1,
        }, timeout=25)
        p = r.json()
    except Exception:
        return None

    rows = p.get('results', [])
    if not rows:
        return None
    x = rows[0]
    return {
        'lat': pd.to_numeric(x.get('LATITUDE'), errors='coerce'),
        'lng': pd.to_numeric(x.get('LONGITUDE'), errors='coerce'),
        'display': x.get('ADDRESS', query),
    }

def to_latlng(df, lat_cols, lng_cols):
    if df is None or len(df) == 0:
        return pd.DataFrame(columns=['lat', 'lng'])
    out = df.copy()
    lat_col = next((c for c in lat_cols if c in out.columns), None)
    lng_col = next((c for c in lng_cols if c in out.columns), None)
    if lat_col is None or lng_col is None:
        return pd.DataFrame(columns=['lat', 'lng'])
    out['lat'] = pd.to_numeric(out[lat_col], errors='coerce')
    out['lng'] = pd.to_numeric(out[lng_col], errors='coerce')
    return out.dropna(subset=['lat', 'lng'])

def nearest_and_count(home_lat, home_lng, poi_coords, radius_km):
    if len(poi_coords) == 0:
        return np.nan, 0
    home = np.radians(np.array([[home_lat, home_lng]], dtype=float))
    tree = BallTree(np.radians(poi_coords.astype(float)), metric='haversine')
    d_rad, _ = tree.query(home, k=1)
    d_km = float(d_rad[0, 0] * 6371.0)
    idx = tree.query_radius(home, r=radius_km / 6371.0)[0]
    return d_km, int(len(idx))

def weighted_mall_access(home_lat, home_lng, mall_geo, radius_km=3.0):
    if mall_geo is None or len(mall_geo) == 0:
        return 0.0
    pts = mall_geo[['lat', 'lng']].to_numpy()
    home = np.radians(np.array([[home_lat, home_lng]], dtype=float))
    tree = BallTree(np.radians(pts.astype(float)), metric='haversine')
    idx_arr, dist_arr = tree.query_radius(home, r=radius_km / 6371.0, return_distance=True, sort_results=True)
    ids = idx_arr[0]
    d = dist_arr[0]
    if len(ids) == 0:
        return 0.0

    names = mall_geo['name'].astype(str).str.upper() if 'name' in mall_geo.columns else pd.Series('', index=mall_geo.index)
    big_kw = names.str.contains('MEGA|HUB|CITY|JUNCTION|POINT|PLAZA|CENTRE', regex=True, na=False)
    w = np.where(big_kw, 1.5, 1.0)
    d_km = np.maximum(d * 6371.0, 0.05)
    return float(np.sum(w[ids] / (d_km + 0.25)))

In [4]:
# Load model, feature columns, and cached geo datasets
model_fp = latest_file(TRAIN_OUTPUT, 'best_price_model_*.joblib')
feature_fp = latest_file(FEATURE_OUTPUT, 'hdb_feature_table_*.csv')

model = joblib.load(model_fp)
feature_df = pd.read_csv(feature_fp)

if 'address_key' not in feature_df.columns:
    raise ValueError('Feature table missing address_key')

feature_df['address_key_norm'] = feature_df['address_key'].astype(str).map(normalize_address)
feature_df['address_key_compact'] = feature_df['address_key_norm'].map(compact)

model_features = getattr(model, 'feature_name_', None)
if model_features is None or len(model_features) == 0:
    model_features = [c for c in feature_df.columns if c not in ['resale_price', 'address_key', 'address_key_norm', 'address_key_compact']]

# POI datasets
hawker_fp = latest_file(GEO_DIR, 'nea_hawker_centres_*.csv')
mall_fp = latest_file(GEO_DIR, 'onemap_mall_nodes_*.csv')
school_geo_fp = latest_file(GEO_DIR, 'moe_school_geocode_*.csv')

hawker_geo = to_latlng(pd.read_csv(hawker_fp), ['lat', 'latitude', 'LATITUDE'], ['lng', 'longitude', 'LONGITUDE', 'longtitude'])
mall_df_raw = pd.read_csv(mall_fp)
mall_geo = to_latlng(mall_df_raw, ['lat', 'LATITUDE'], ['lng', 'LONGITUDE'])
if len(mall_geo):
    if 'name' in mall_df_raw.columns:
        mall_geo['name'] = mall_df_raw['name'].astype(str).values[:len(mall_geo)]
    elif 'SEARCHVAL' in mall_df_raw.columns:
        mall_geo['name'] = mall_df_raw['SEARCHVAL'].astype(str).values[:len(mall_geo)]
    else:
        mall_geo['name'] = ''

school_geo = to_latlng(pd.read_csv(school_geo_fp), ['lat'], ['lng'])

# Build primary school quality map
moe_fp = latest_file(SCHOOL_DIR, 'moe_general_information_of_schools_*.csv')
sg_fp = latest_file(SCHOOL_DIR, 'sgschooling_2015plus_*.csv')
moe = pd.read_csv(moe_fp)
sg = pd.read_csv(sg_fp)

s = sg.copy()
s = s[~s['school'].astype(str).str.startswith('↳', na=False)].copy()
for c in ['competition_ratio_extracted', 'phase_1', 'applicants_extracted', 'vacancies_extracted']:
    if c in s.columns:
        s[c] = pd.to_numeric(s[c], errors='coerce')
comp = s.get('competition_ratio_extracted', pd.Series(np.nan, index=s.index))
phase1 = s.get('phase_1', pd.Series(np.nan, index=s.index))
apps = s.get('applicants_extracted', pd.Series(np.nan, index=s.index))
vac = s.get('vacancies_extracted', pd.Series(np.nan, index=s.index)).replace(0, np.nan)
raw_q = np.nanmax(np.vstack([comp.fillna(np.nan), (apps / vac).fillna(np.nan)]), axis=0)
raw_q = pd.Series(raw_q, index=s.index).fillna(1.0)
p1 = phase1.fillna(phase1.median() if phase1.notna().any() else 0.0)
q = 0.7 * raw_q + 0.3 * (p1 / (p1.max() if p1.max() else 1.0))
pq = pd.DataFrame({'school_upper': s['school'].astype(str).str.upper().str.strip(), 'q': q}).groupby('school_upper', as_index=False)['q'].mean()
qmin, qmax = float(pq['q'].min()), float(pq['q'].max())
pq['school_quality_score'] = 100 * (pq['q'] - qmin) / (qmax - qmin + 1e-9)

moe['school_upper'] = moe['school_name'].astype(str).str.upper().str.strip()
primary = moe[moe['mainlevel_code'].astype(str).str.upper() == 'PRIMARY'][['school_name', 'school_upper']].copy()
sg_geo = pd.read_csv(school_geo_fp)
sg_geo['school_name'] = sg_geo['school_name'].astype(str)
sg_geo = sg_geo.dropna(subset=['lat', 'lng']).copy()
primary_geo = primary.merge(pq, on='school_upper', how='left').merge(sg_geo[['school_name', 'lat', 'lng']], on='school_name', how='left')
primary_geo['lat'] = pd.to_numeric(primary_geo['lat'], errors='coerce')
primary_geo['lng'] = pd.to_numeric(primary_geo['lng'], errors='coerce')
primary_geo = primary_geo.dropna(subset=['lat', 'lng']).copy()

# KNN proxy table for orientation/highway/mrt from known geocoded HDB points
acc_fp = latest_file(GEO_DIR, 'hdb_geo_accessibility_noise_features_*.csv')
acc = pd.read_csv(acc_fp)
known = acc[['source_id', 'lat', 'lng', 'nearest_mrt_km', 'road_noise_score', 'facing_road_noise_proxy']].copy()
known['lat'] = pd.to_numeric(known['lat'], errors='coerce')
known['lng'] = pd.to_numeric(known['lng'], errors='coerce')
known['dist_to_mrt_m'] = pd.to_numeric(known['nearest_mrt_km'], errors='coerce') * 1000
known = known.dropna(subset=['lat', 'lng'])

hw_fp = latest_file(GEO_DIR, 'onemap_hdb_geocode_with_highway_dist_*.csv')
hw = pd.read_csv(hw_fp)
hw = hw[['source_id', 'highway_distance_km']].copy()
hw['dist_to_highway_m'] = pd.to_numeric(hw['highway_distance_km'], errors='coerce') * 1000
known = known.merge(hw[['source_id', 'dist_to_highway_m']], on='source_id', how='left')

frp = pd.to_numeric(known['facing_road_noise_proxy'], errors='coerce')
rns = pd.to_numeric(known['road_noise_score'], errors='coerce')
known['orientation_score'] = np.where(frp.notna(), np.where(frp > 0.5, -1.0, 1.0), np.nan)
known.loc[known['orientation_score'].isna(), 'orientation_score'] = np.where(rns.notna(), np.where(rns > rns.median(), -1.0, 1.0), np.nan)

known_geo = known[['lat', 'lng']].to_numpy()
known_tree = BallTree(np.radians(known_geo.astype(float)), metric='haversine')

print('Loaded model:', model_fp.name)
print('Model features:', len(model_features))
print('Known geo rows for proxy:', len(known))

Loaded model: best_price_model_20260317.joblib
Model features: 69
Known geo rows for proxy: 9710


/var/folders/6p/nwg1ljcj77bfnrwnqw_yprf80000gn/T/ipykernel_23067/2172305074.py:51: RuntimeWarning: All-NaN slice encountered
  raw_q = np.nanmax(np.vstack([comp.fillna(np.nan), (apps / vac).fillna(np.nan)]), axis=0)


In [5]:
def activate_onehot(x_row: pd.DataFrame, prefix: str, value: str):
    if value is None:
        return False
    value_norm = compact(value)
    candidates = [c for c in x_row.columns if c.startswith(prefix + '_')]
    for c in candidates:
        suffix = c[len(prefix) + 1:]
        if compact(suffix) == value_norm:
            x_row.loc[:, c] = 1
            return True
    return False

def nearest_known_proxies(lat: float, lng: float, k: int = 30):
    home = np.radians(np.array([[lat, lng]], dtype=float))
    d_rad, idx = known_tree.query(home, k=min(k, len(known)))
    near = known.iloc[idx[0]].copy()
    d_km = np.maximum(d_rad[0] * 6371.0, 0.03)
    w = 1.0 / d_km

    out = {}
    for c in ['dist_to_mrt_m', 'dist_to_highway_m', 'orientation_score']:
        s = pd.to_numeric(near[c], errors='coerce')
        if s.notna().any():
            out[c] = float(np.average(s.fillna(s.median()), weights=w))
        else:
            out[c] = np.nan
    out['orientation_score'] = 1.0 if out['orientation_score'] >= 0 else -1.0
    return out

def weighted_mall_access_safe(lat: float, lng: float):
    if mall_geo is None or len(mall_geo) == 0:
        return 0.0
    return weighted_mall_access(lat, lng, mall_geo[['lat', 'lng', 'name']].drop_duplicates(), radius_km=3.0)

def build_single_row_from_address(unit_address_input: str, unit_profile_input: dict):
    # unit_profile_input is authoritative structural input from current unit info.
    required = ['floor_area_sqm', 'room_count', 'lease_commence_year', 'flat_type', 'flat_model', 'town']
    missing = [k for k in required if k not in unit_profile_input or unit_profile_input[k] in [None, '']]
    if missing:
        raise ValueError(f'Missing required unit profile fields: {missing}')

    addr_norm = normalize_address(unit_address_input)

    geocode = onemap_geocode(addr_norm)
    if geocode is None or pd.isna(geocode['lat']) or pd.isna(geocode['lng']):
        raise ValueError('Unable to geocode address via OneMap. Please refine the address string.')

    lat, lng = float(geocode['lat']), float(geocode['lng'])

    # Structural features from current unit input (not from historical transactions)
    tx_year = datetime.now().year
    lease_commence = float(unit_profile_input['lease_commence_year'])
    lease_remaining_years = float(np.clip(99 - (tx_year - lease_commence), 0, 99))

    floor_from_unit = parse_unit_floor(unit_address_input)
    level_mid = float(floor_from_unit) if floor_from_unit is not None else float(unit_profile_input.get('level_mid', np.nan))

    # API/geo-derived features
    dist_to_food_km, _ = nearest_and_count(lat, lng, hawker_geo[['lat', 'lng']].drop_duplicates().to_numpy(), 1.0)
    dist_to_mall_km, mall_count_3km = nearest_and_count(lat, lng, mall_geo[['lat', 'lng']].drop_duplicates().to_numpy(), 3.0)
    dist_to_school_km, school_count_1km = nearest_and_count(lat, lng, school_geo[['lat', 'lng']].drop_duplicates().to_numpy(), 1.0)

    # Primary school quality within 1km
    pcoords = primary_geo[['lat', 'lng']].to_numpy() if len(primary_geo) else np.empty((0, 2))
    pqual = primary_geo['school_quality_score'].fillna(primary_geo['school_quality_score'].median() if primary_geo['school_quality_score'].notna().any() else 50.0).to_numpy() if len(primary_geo) else np.array([])
    if len(pcoords):
        tree = BallTree(np.radians(pcoords.astype(float)), metric='haversine')
        home = np.radians(np.array([[lat, lng]], dtype=float))
        ids, d = tree.query_radius(home, r=1.0 / 6371.0, return_distance=True, sort_results=True)
        ids = ids[0]
        d = d[0]
        if len(ids):
            w = 1.0 / np.maximum(d * 6371.0, 0.05)
            q = pqual[ids]
            primary_quality_weighted = float(np.average(q, weights=w))
            primary_top_quality = float(np.max(q))
            primary_count = int(len(ids))
        else:
            primary_quality_weighted = np.nan
            primary_top_quality = np.nan
            primary_count = 0
    else:
        primary_quality_weighted = np.nan
        primary_top_quality = np.nan
        primary_count = 0

    prox = nearest_known_proxies(lat, lng, k=40)

    feats = {
        'transaction_year': float(tx_year),
        'level_mid': 0.0 if pd.isna(level_mid) else float(level_mid),
        'lease_remaining_years': lease_remaining_years,
        'floor_area_sqm': float(unit_profile_input['floor_area_sqm']),
        'room_count': float(unit_profile_input['room_count']),
        'dist_to_mrt_m': prox['dist_to_mrt_m'],
        'orientation_score': prox['orientation_score'],
        'dist_to_highway_m': prox['dist_to_highway_m'],
        'dist_to_foodcourt_m': float(dist_to_food_km * 1000) if not pd.isna(dist_to_food_km) else np.nan,
        'dist_to_nearest_mall_m': float(dist_to_mall_km * 1000) if not pd.isna(dist_to_mall_km) else np.nan,
        'mall_count_3km': float(mall_count_3km),
        'mall_weighted_access_3km': float(weighted_mall_access_safe(lat, lng)),
        'dist_to_nearest_school_m': float(dist_to_school_km * 1000) if not pd.isna(dist_to_school_km) else np.nan,
        'school_count_1km': float(school_count_1km),
        'primary_school_quality_1km_weighted': primary_quality_weighted,
        'primary_school_top_quality_1km': primary_top_quality,
        'primary_school_count_1km': float(primary_count),
    }

    x_row = pd.DataFrame(0.0, index=[0], columns=model_features)
    for k, v in feats.items():
        if k in x_row.columns:
            x_row.loc[0, k] = 0.0 if pd.isna(v) else float(v)

    town_hit = activate_onehot(x_row, 'town', str(unit_profile_input['town']))
    flat_type_hit = activate_onehot(x_row, 'flat_type', str(unit_profile_input['flat_type']))
    flat_model_hit = activate_onehot(x_row, 'flat_model', str(unit_profile_input['flat_model']))

    info = {
        'input_address': unit_address_input,
        'normalized_address': addr_norm,
        'geocode_display': geocode.get('display'),
        'lat': lat,
        'lng': lng,
        'unit_profile_input': unit_profile_input,
        'onehot_matched': {
            'town': bool(town_hit),
            'flat_type': bool(flat_type_hit),
            'flat_model': bool(flat_model_hit),
        },
    }
    return x_row, info

def feature_group(name: str) -> str:
    school_set = {
        'dist_to_nearest_school_m', 'school_count_1km',
        'primary_school_quality_1km_weighted', 'primary_school_top_quality_1km', 'primary_school_count_1km'
    }
    if name in school_set:
        return 'school_all_factors'
    if name.startswith('town_'):
        return 'town(one-hot)'
    if name.startswith('flat_type_'):
        return 'flat_type(one-hot)'
    if name.startswith('flat_model_'):
        return 'flat_model(one-hot)'
    return name

def predict_and_explain(unit_address_input: str, unit_profile_input: dict):
    x_row, info = build_single_row_from_address(unit_address_input, unit_profile_input)
    pred_price = float(model.predict(x_row)[0])

    contrib = model.booster_.predict(x_row, pred_contrib=True)
    shap_vals = np.array(contrib[0][:-1], dtype=float)
    base_value = float(contrib[0][-1])

    out = pd.DataFrame({
        'feature': model_features,
        'feature_value': x_row.iloc[0].values,
        'shap_value': shap_vals,
    })
    out['abs_shap'] = out['shap_value'].abs()
    out = out.sort_values('abs_shap', ascending=False).reset_index(drop=True)

    total_abs = float(out['abs_shap'].sum()) if float(out['abs_shap'].sum()) > 0 else 1.0
    out['pct_of_total_abs'] = 100.0 * out['abs_shap'] / total_abs

    top20 = out.head(20).copy()
    top20_abs = float(top20['abs_shap'].sum()) if float(top20['abs_shap'].sum()) > 0 else 1.0
    top20['pct_within_top20_abs'] = 100.0 * top20['abs_shap'] / top20_abs

    grp = out.copy()
    grp['feature_group'] = grp['feature'].map(feature_group)
    grp = grp.groupby('feature_group', as_index=False).agg(
        contribution=('shap_value', 'sum'),
        abs_contribution=('abs_shap', 'sum')
    ).sort_values('abs_contribution', ascending=False).reset_index(drop=True)
    grp['pct_of_total_abs'] = 100.0 * grp['abs_contribution'] / total_abs

    safe = re.sub(r'[^A-Z0-9]+', '_', info['normalized_address'])[:60]
    raw_fp = TEST_OUTPUT / f'unit_top20_shap_{RUN_DATE}_{safe}.csv'
    grp_fp = TEST_OUTPUT / f'unit_group_shap_{RUN_DATE}_{safe}.csv'
    info_fp = TEST_OUTPUT / f'unit_inference_info_{RUN_DATE}_{safe}.json'

    top20.to_csv(raw_fp, index=False)
    grp.to_csv(grp_fp, index=False)
    with open(info_fp, 'w', encoding='utf-8') as f:
        json.dump({**info, 'predicted_price': pred_price, 'base_value': base_value}, f, indent=2)

    print('Input address:', info['input_address'])
    print('Normalized:', info['normalized_address'])
    print('Geocode display:', info['geocode_display'])
    print('Predicted resale price:', round(pred_price, 2))
    print('SHAP base value:', round(base_value, 2))
    print('Check sum (base + sum(shap)):', round(base_value + float(np.sum(shap_vals)), 2))
    print('Onehot matched:', info['onehot_matched'])
    print('Saved top20:', raw_fp.name)
    print('Saved grouped:', grp_fp.name)
    print('Saved info:', info_fp.name)

    print('\nTop20 factors with percentage:')
    display(top20[['feature', 'feature_value', 'shap_value', 'pct_of_total_abs', 'pct_within_top20_abs']])

    print('\nGrouped factor contribution:')
    display(grp.head(20))

    return pred_price, top20, grp, info

In [6]:
# Input address + current unit profile here (authoritative unit info)
unit_address_input = '441B Clementi Avenue 3 #12-12'
unit_profile_input = {
    'floor_area_sqm': 1033,
    'room_count': 5,
    'lease_commence_year': 2011,
    'flat_type': '3 ROOM',
    'flat_model': 'Improved',
    'town': 'CLEMENTI',
    # Optional: if unit number is unavailable in address, you can still set level_mid manually
    # 'level_mid': 3,
}

pred_price, top20_df, group_df, info = predict_and_explain(unit_address_input, unit_profile_input)

Input address: 441B Clementi Avenue 3 #12-12
Normalized: 441B CLEMENTI AVENUE 3
Geocode display: 441B CLEMENTI AVENUE 3 CLEMENTI TOWERS SINGAPORE 122441
Predicted resale price: 1389740.07
SHAP base value: 512309.52
Check sum (base + sum(shap)): 1389740.07
Onehot matched: {'town': True, 'flat_type': True, 'flat_model': True}
Saved top20: unit_top20_shap_20260329_441B_CLEMENTI_AVENUE_3.csv
Saved grouped: unit_group_shap_20260329_441B_CLEMENTI_AVENUE_3.csv
Saved info: unit_inference_info_20260329_441B_CLEMENTI_AVENUE_3.json

Top20 factors with percentage:


,feature,feature_value,shap_value,pct_of_total_abs,pct_within_top20_abs
0,floor_area_sqm,1033.000000,392509.331764,42.430748,43.394133
1,transaction_year,2026.000000,191385.109957,20.688969,21.158710
2,dist_to_mrt_m,2976.508939,52023.590765,5.623815,5.751503
3,room_count,5.000000,51000.806994,5.513251,5.638429
4,dist_to_nearest_mall_m,76.136567,39953.937699,4.319070,4.417134
5,mall_weighted_access_3km,9.894960,38560.376277,4.168425,4.263068
6,dist_to_foodcourt_m,145.285694,28654.066473,3.097540,3.167870
7,town_CLEMENTI,1.000000,21627.539865,2.337964,2.391047
8,town_WOODLANDS,0.000000,14343.568629,1.550558,1.585763
9,lease_remaining_years,84.000000,12812.389165,1.385035,1.416482



Grouped factor contribution:


,feature_group,contribution,abs_contribution,pct_of_total_abs
0,floor_area_sqm,392509.331764,392509.331764,42.430748
1,transaction_year,191385.109957,191385.109957,20.688969
2,town(one-hot),46541.134581,64217.734815,6.942017
3,dist_to_mrt_m,52023.590765,52023.590765,5.623815
4,room_count,51000.806994,51000.806994,5.513251
5,dist_to_nearest_mall_m,39953.937699,39953.937699,4.319070
6,mall_weighted_access_3km,38560.376277,38560.376277,4.168425
7,dist_to_foodcourt_m,28654.066473,28654.066473,3.097540
8,flat_type(one-hot),9208.366256,17531.922919,1.895223
9,lease_remaining_years,12812.389165,12812.389165,1.385035


In [7]:
# Diagnostic cell: print the key intermediate values from the latest prediction

# Recompute the intermediate features for easier inspection

_lat, _lng = info['lat'], info['lng']



print("=== Geolocation ===")

print(f"  lat={_lat:.6f}, lng={_lng:.6f}")



print("\n=== Hawker centre distance ===")

_d_food_km, _n_food = nearest_and_count(_lat, _lng, hawker_geo[['lat','lng']].drop_duplicates().to_numpy(), 1.0)

print(f"  Nearest NEA hawker centre: {_d_food_km*1000:.1f} m  (count within 1 km: {_n_food})")

# Show the nearest 3 hawker centres

from sklearn.neighbors import BallTree as _BT

import numpy as _np2

_hk_coords = hawker_geo[['lat','lng']].drop_duplicates().to_numpy()

_htree = _BT(_np2.radians(_hk_coords.astype(float)), metric='haversine')

_hd, _hi = _htree.query(_np2.radians([[_lat, _lng]]), k=min(3, len(_hk_coords)))

_hd_m = _hd[0] * 6371000

print("  Nearest 3 NEA hawker centres:")

for _rank, (_di, _ii) in enumerate(zip(_hd_m, _hi[0])):

    _row = hawker_geo.iloc[_ii]

    _name = _row.get('name', _row.get('NAME', _row.get('description', str(_ii))))

    print(f"    {_rank+1}. {_name}  distance {_di:.1f} m")



print("\n=== Expressway distance (proxy BallTree) ===")

_prox = nearest_known_proxies(_lat, _lng, k=40)

print(f"  dist_to_highway_m (weighted proxy) = {_prox.get('dist_to_highway_m'):.1f} m")

print(f"  orientation_score (binarized)       = {_prox.get('orientation_score')}")

print(f"  dist_to_mrt_m (weighted proxy)      = {_prox.get('dist_to_mrt_m'):.1f} m")



# Inspect the raw values of nearby proxy points

_home = _np2.radians(_np2.array([[_lat, _lng]], dtype=float))

_dr, _idx = known_tree.query(_home, k=10)

_near10 = known.iloc[_idx[0]][['lat','lng','dist_to_highway_m','orientation_score','dist_to_mrt_m']].copy()

_near10['dist_to_point_m'] = _dr[0] * 6371000

print("\n  Raw values of the nearest 10 proxy points:")

print(_near10.to_string(index=False))



print("\n=== School features ===")

_pcoords = primary_geo[['lat','lng']].to_numpy()

_ptree = _BT(_np2.radians(_pcoords.astype(float)), metric='haversine')

_pd, _pi = _ptree.query(_np2.radians([[_lat, _lng]]), k=min(5, len(_pcoords)))

_pd_m = _pd[0] * 6371000

print(f"  Nearest 5 primary schools:")

for _rank, (_di, _ii) in enumerate(zip(_pd_m, _pi[0])):

    _row = primary_geo.iloc[_ii]

    _sname = _row.get('school_name', str(_ii))

    _sq = _row.get('school_quality_score', float('nan'))

    print(f"    {_rank+1}. {_sname}  distance {_di:.1f} m  quality score={_sq:.2f}")



# Primary schools within 1 km

_ids1km, _d1km = _ptree.query_radius(_np2.radians([[_lat, _lng]]), r=1.0/6371.0, return_distance=True, sort_results=True)

_ids1km = _ids1km[0]; _d1km_m = _d1km[0] * 6371000

print(f"\n  Primary school count within 1 km: {len(_ids1km)}")

for _ii2, _di2 in zip(_ids1km, _d1km_m):

    _row2 = primary_geo.iloc[_ii2]

    _sname2 = _row2.get('school_name', str(_ii2))

    _sq2 = _row2.get('school_quality_score', float('nan'))

    print(f"    - {_sname2}  {_di2:.1f} m  quality score={_sq2:.2f}")


=== Geolocation ===
  lat=1.314576, lng=103.764159

=== Hawker centre distance ===
  Nearest NEA hawker centre: 145.3 m  (count within 1 km: 2)
  Nearest 3 NEA hawker centres:
    1. MARKET & HAWKER CENTRE (BLK 448 CLEMENTI AVENUE 3)  distance 145.3 m
    2. MARKET & HAWKER CENTRE (BLK 353 CLEMENTI AVE 2)  distance 751.9 m
    3. MARKET & HAWKER CENTRE (BLK 726 CLEMENTI WEST STREET 2)  distance 1197.1 m

=== Expressway distance (proxy BallTree) ===
  dist_to_highway_m (weighted proxy) = 2264.6 m
  orientation_score (binarized)       = -1.0
  dist_to_mrt_m (weighted proxy)      = 2976.5 m

  Raw values of the nearest 10 proxy points:
     lat        lng  dist_to_highway_m  orientation_score  dist_to_mrt_m  dist_to_point_m
1.314576 103.764159        2259.298400               -1.0    3069.613953         0.027216
1.314211 103.764010        2218.687958               -1.0    3062.111357        43.853259
1.314123 103.764519        2251.871833               -1.0    3008.885704        64.360607

In [8]:
# Quick inspection: compare target primary schools and current POI sizing/quality proxies
import pandas as pd

school_cmp = primary_geo[['school_name', 'school_quality_score']].copy()
school_cmp['school_upper'] = school_cmp['school_name'].astype(str).str.upper().str.strip()

target_pattern = 'CLEMENTI PRIMARY|NAN HUA PRIMARY|PARK VIEW PRIMARY|ELIAS PARK PRIMARY'
target = school_cmp[school_cmp['school_upper'].str.contains(target_pattern, regex=True, na=False)]

print('=== Target primary school quality_score ===')
print(target[['school_name', 'school_quality_score']].drop_duplicates().sort_values('school_quality_score', ascending=False).to_string(index=False))

print('\n=== Hawker dataset columns (for size/quality clues) ===')
print(sorted(hawker_geo.columns.tolist()))

print('\n=== Mall dataset columns (for size/quality clues) ===')
print(sorted(mall_geo.columns.tolist()))
if 'name' in mall_geo.columns:
    sample = mall_geo['name'].dropna().astype(str).head(20)
    print('\nMall name sample:')
    print(sample.to_string(index=False))



=== Target primary school quality_score ===
              school_name  school_quality_score
  CLEMENTI PRIMARY SCHOOL                   NaN
ELIAS PARK PRIMARY SCHOOL                   NaN
   NAN HUA PRIMARY SCHOOL                   NaN
 PARK VIEW PRIMARY SCHOOL                   NaN

=== Hawker dataset columns (for size/quality clues) ===
['BLK_NO', 'BUILDING', 'POSTAL', 'ROAD_NAME', 'X', 'Y', 'address', 'lat', 'lng', 'name']

=== Mall dataset columns (for size/quality clues) ===
['BLK_NO', 'BUILDING', 'POSTAL', 'ROAD_NAME', 'X', 'Y', 'address', 'lat', 'lng', 'name', 'source']

Mall name sample:
                          NEX
myVillage at Serangoon Garden
                      AMK Hub
                   Junction 8
              The Poiz Centre
         Heartland Mall Kovan
                 Hougang Mall
                Nex Serangoon
                  Compass One
               Waterway Point
               Rivervale Mall
               Oasis Terraces
              Northpoint City
       

In [9]:
# Debug name matching for Clementi Primary / Nan Hua Primary / Park View Primary / Elias Park Primary
print('\n=== Name matching debug in pq (raw quality table) ===')
_pq_hit = pq[pq['school_upper'].str.contains('CLEMENTI|NAN HUA|PARK VIEW|ELIAS PARK', regex=True, na=False)].copy()
print(_pq_hit.sort_values('school_upper').to_string(index=False))

print('\n=== Name matching debug in primary_geo merged table ===')
_pg_hit = primary_geo[primary_geo['school_name'].astype(str).str.upper().str.contains('CLEMENTI|NAN HUA|PARK VIEW|ELIAS PARK', regex=True, na=False)].copy()
show_cols = [c for c in ['school_name', 'school_upper', 'q', 'school_quality_score', 'lat', 'lng'] if c in _pg_hit.columns]
print(_pg_hit[show_cols].sort_values('school_name').to_string(index=False))




=== Name matching debug in pq (raw quality table) ===
school_upper        q  school_quality_score
    CLEMENTI 0.811234             35.786130
  ELIAS PARK 0.805844             31.392271
     NAN HUA 0.834286             54.579142
   PARK VIEW 0.791558             19.745897

=== Name matching debug in primary_geo merged table ===
              school_name              school_upper   q  school_quality_score      lat        lng
  CLEMENTI PRIMARY SCHOOL   CLEMENTI PRIMARY SCHOOL NaN                   NaN 1.315063 103.763144
ELIAS PARK PRIMARY SCHOOL ELIAS PARK PRIMARY SCHOOL NaN                   NaN 1.375057 103.945289
   NAN HUA PRIMARY SCHOOL    NAN HUA PRIMARY SCHOOL NaN                   NaN 1.319202 103.761095
 PARK VIEW PRIMARY SCHOOL  PARK VIEW PRIMARY SCHOOL NaN                   NaN 1.378017 103.939202
